# US Birth-to-Death Dispersal & Out-Migration

This notebook measures **the propensity to leave your home state** for people in Wikidata: compare birth state to death state, and the great-circle distance between those places.

**Cohort.** Humans (`wd:Q5`) with a recorded death date on or after `1980-01-01`, so the sample reflects contemporary mobility rather than historical frontier settlement.

**Sample cap.** Up to 5,000 most recent deaths per US state of birth (`50 × 5,000 = 250,000` potential rows). Wikidata is a catalog of *notable* people, not a random sample of residents. Rates describe this graph, not the US population.

**Birth state.** The inner SPARQL subquery filters with `wdt:P131*` to the state's Wikidata item. Immediate `P131` is usually a county, so birth state is stamped from the state being queried rather than from a one-hop label. Death state is resolved in the outer query with `P131*` to an item that is an instance of US state (`wd:Q35657`).

## How to run

1. `python -m venv .venv && source .venv/bin/activate && pip install -r requirements.txt`
2. `RUN_FULL_EXTRACT` is `True`: running all cells queries every US state and writes `./data/{State}.csv`. Files already on disk are skipped, so a run can resume after a timeout.
3. Set `RUN_FULL_EXTRACT = False` for a one-state smoke test only (`Rhode Island`, 20 rows, under `./data/_smoke/`).

Queries hit `https://query.wikidata.org/sparql` with a 90s pause (Wikidata's one-request-per-minute limit), retries, and a custom User-Agent. A full 50-state pull can take a long time and may still time out on populous states.


## Setup

In [1]:
from __future__ import annotations

import json
import time
from pathlib import Path
from urllib.error import HTTPError, URLError

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from SPARQLWrapper import JSON, SPARQLWrapper
from SPARQLWrapper.SPARQLExceptions import SPARQLWrapperException

sns.set_theme(style="whitegrid")
pd.set_option("display.max_rows", 60)
pd.set_option("display.max_columns", 20)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [2]:
WIKIDATA_SPARQL = "https://query.wikidata.org/sparql"
USER_AGENT = "NatalDispersalResearch/1.0 (US birth-death dispersal notebook)"
QUERY_TIMEOUT_S = 120
RATE_LIMIT_S = 90.0
MAX_RETRIES = 3
FULL_LIMIT = 5000
SMOKE_LIMIT = 20
SMOKE_STATE = "Rhode Island"

DATA_DIR = Path("./data")
SMOKE_DIR = DATA_DIR / "_smoke"
FAILURES_PATH = DATA_DIR / "_failures.json"

# Set False to skip the Wikidata pull and use CSVs already in ./data/.
RUN_FULL_EXTRACT = True

# Q-codes from https://www.wikidata.org/wiki/Wikidata:Lists/US_states
# (state items only; not cities, countries, or DC).
STATES: dict[str, dict[str, str]] = {
    "Alabama": {"qid": "Q173", "code": "AL"},
    "Alaska": {"qid": "Q797", "code": "AK"},
    "Arizona": {"qid": "Q816", "code": "AZ"},
    "Arkansas": {"qid": "Q1612", "code": "AR"},
    "California": {"qid": "Q99", "code": "CA"},
    "Colorado": {"qid": "Q1261", "code": "CO"},
    "Connecticut": {"qid": "Q779", "code": "CT"},
    "Delaware": {"qid": "Q1393", "code": "DE"},
    "Florida": {"qid": "Q812", "code": "FL"},
    "Georgia": {"qid": "Q1428", "code": "GA"},
    "Hawaii": {"qid": "Q782", "code": "HI"},
    "Idaho": {"qid": "Q1221", "code": "ID"},
    "Illinois": {"qid": "Q1204", "code": "IL"},
    "Indiana": {"qid": "Q1415", "code": "IN"},
    "Iowa": {"qid": "Q1546", "code": "IA"},
    "Kansas": {"qid": "Q1558", "code": "KS"},
    "Kentucky": {"qid": "Q1603", "code": "KY"},
    "Louisiana": {"qid": "Q1588", "code": "LA"},
    "Maine": {"qid": "Q724", "code": "ME"},
    "Maryland": {"qid": "Q1391", "code": "MD"},
    "Massachusetts": {"qid": "Q771", "code": "MA"},
    "Michigan": {"qid": "Q1166", "code": "MI"},
    "Minnesota": {"qid": "Q1527", "code": "MN"},
    "Mississippi": {"qid": "Q1494", "code": "MS"},
    "Missouri": {"qid": "Q1581", "code": "MO"},
    "Montana": {"qid": "Q1212", "code": "MT"},
    "Nebraska": {"qid": "Q1553", "code": "NE"},
    "Nevada": {"qid": "Q1227", "code": "NV"},
    "New Hampshire": {"qid": "Q759", "code": "NH"},
    "New Jersey": {"qid": "Q1408", "code": "NJ"},
    "New Mexico": {"qid": "Q1522", "code": "NM"},
    "New York": {"qid": "Q1384", "code": "NY"},
    "North Carolina": {"qid": "Q1454", "code": "NC"},
    "North Dakota": {"qid": "Q1207", "code": "ND"},
    "Ohio": {"qid": "Q1397", "code": "OH"},
    "Oklahoma": {"qid": "Q1649", "code": "OK"},
    "Oregon": {"qid": "Q824", "code": "OR"},
    "Pennsylvania": {"qid": "Q1400", "code": "PA"},
    "Rhode Island": {"qid": "Q1387", "code": "RI"},
    "South Carolina": {"qid": "Q1456", "code": "SC"},
    "South Dakota": {"qid": "Q1211", "code": "SD"},
    "Tennessee": {"qid": "Q1509", "code": "TN"},
    "Texas": {"qid": "Q1439", "code": "TX"},
    "Utah": {"qid": "Q829", "code": "UT"},
    "Vermont": {"qid": "Q16551", "code": "VT"},
    "Virginia": {"qid": "Q1370", "code": "VA"},
    "Washington": {"qid": "Q1223", "code": "WA"},
    "West Virginia": {"qid": "Q1371", "code": "WV"},
    "Wisconsin": {"qid": "Q1537", "code": "WI"},
    "Wyoming": {"qid": "Q1214", "code": "WY"},
}

assert len(STATES) == 50, f"Expected 50 states, got {len(STATES)}"
DATA_DIR.mkdir(parents=True, exist_ok=True)
SMOKE_DIR.mkdir(parents=True, exist_ok=True)

## Section 1: Extraction & Caching

For each state, the **inner** subquery is the specified modern-cohort filter: birth place in the state (`P131*`), human, death place and date, `FILTER` on 1980-01-01, newest deaths first, `LIMIT` applied **before** labels and coordinates.

The **outer** query reads death country, coordinates, and US death state. Death state uses `P131*` from the already-bound death place to an item that is a US state (`wd:Q35657`). If that column comes back empty, a follow-up `VALUES` query resolves it. A Blazegraph reverse-gearing hint on the inner `P131*` keeps that specified triple but starts the path from the state item.

Cached output: `./data/{state_name}.csv`. Existing files are skipped so the loop can resume after a timeout.

In [3]:
QUERY_TEMPLATE = """
SELECT DISTINCT
  ?person
  ?personLabel
  ?deathDate
  ?deathPlace
  ?deathCountry
  ?deathCountryLabel
  ?birthLat
  ?birthLon
  ?deathLat
  ?deathLon
  ?deathState
  ?deathStateLabel
WHERE {
  {
    SELECT ?person ?birthPlace ?deathPlace ?deathDate WHERE {
      ?birthPlace wdt:P131* wd:{STATE_QCODE} .
      hint:Prior hint:gearing "reverse" .
      ?person wdt:P19 ?birthPlace ;
              wdt:P31 wd:Q5 ;
              wdt:P20 ?deathPlace ;
              wdt:P570 ?deathDate .
      FILTER(?deathDate >= "1980-01-01T00:00:00Z"^^xsd:dateTime)
    }
    ORDER BY DESC(?deathDate)
    LIMIT {LIMIT}
  }

  OPTIONAL { ?deathPlace wdt:P17 ?deathCountry . }
  OPTIONAL { ?birthPlace wdt:P625 ?birthLocation . }
  OPTIONAL { ?deathPlace wdt:P625 ?deathLocation . }
  OPTIONAL {
    ?deathPlace wdt:P131* ?deathState .
    hint:Prior hint:gearing "forward" .
    ?deathState wdt:P31 wd:Q35657 .
  }

  BIND(geof:latitude(?birthLocation) AS ?birthLat)
  BIND(geof:longitude(?birthLocation) AS ?birthLon)
  BIND(geof:latitude(?deathLocation) AS ?deathLat)
  BIND(geof:longitude(?deathLocation) AS ?deathLon)

  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
}
"""

DEATH_STATE_QUERY = """
SELECT ?deathPlace ?deathState ?deathStateLabel WHERE {
  VALUES ?deathPlace { %s }
  OPTIONAL {
    ?deathPlace wdt:P131* ?deathState .
    hint:Prior hint:gearing "forward" .
    ?deathState wdt:P31 wd:Q35657 .
  }
  SERVICE wikibase:label { bd:serviceParam wikibase:language "en". }
}
"""

PERSON_COLUMNS = (
    "person",
    "personLabel",
    "deathDate",
    "deathPlace",
    "deathCountry",
    "deathCountryLabel",
    "birthLat",
    "birthLon",
    "deathLat",
    "deathLon",
    "deathState",
    "deathStateLabel",
)
PLACE_CHUNK = 80


def build_query(state_qid: str, limit: int) -> str:
    if not state_qid.startswith("Q") or not state_qid[1:].isdigit():
        raise ValueError(f"Invalid Wikidata Q-code: {state_qid}")
    return QUERY_TEMPLATE.replace("{STATE_QCODE}", state_qid).replace("{LIMIT}", str(int(limit)))


def bindings_to_frame(bindings: list[dict], keys: tuple[str, ...]) -> pd.DataFrame:
    rows = [{key: binding.get(key, {}).get("value") for key in keys} for binding in bindings]
    return pd.DataFrame(rows, columns=list(keys))


def qid_from_uri(uri: object) -> str | None:
    if not isinstance(uri, str) or not uri:
        return None
    qid = uri.rsplit("/", 1)[-1]
    if qid.startswith("Q") and qid[1:].isdigit():
        return qid
    return None


def load_failures() -> dict:
    if FAILURES_PATH.exists():
        return json.loads(FAILURES_PATH.read_text(encoding="utf-8"))
    return {}


def save_failures(failures: dict) -> None:
    FAILURES_PATH.write_text(json.dumps(failures, indent=2, sort_keys=True), encoding="utf-8")


def query_wikidata(sparql_query: str) -> dict:
    """GET a SPARQL query with timeout retries and 429/5xx backoff."""
    client = SPARQLWrapper(WIKIDATA_SPARQL, agent=USER_AGENT)
    client.setReturnFormat(JSON)
    client.setTimeout(QUERY_TIMEOUT_S)
    client.setMethod("GET")
    client.setQuery(sparql_query)

    delay = 2.0
    last_error: Exception | None = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            result = client.query()
            raw = result.response.read().decode("utf-8")
            return json.loads(raw, strict=False)
        except json.JSONDecodeError as exc:
            last_error = exc
            if attempt == MAX_RETRIES:
                raise
            print(f"  JSONDecodeError: {exc}; retry {attempt}/{MAX_RETRIES} in {delay:.1f}s", flush=True)
            time.sleep(delay)
            delay *= 2
            continue
        except HTTPError as exc:
            last_error = exc
            retryable = exc.code in {429, 500, 502, 503, 504}
            if not retryable or attempt == MAX_RETRIES:
                raise
            sleep_for = delay
            retry_after = exc.headers.get("Retry-After") if exc.headers else None
            message = str(exc)
            if exc.code == 429:
                # fetch_state falls back to QLever. Retrying here just extends the WDQS ban.
                raise
            if retry_after and str(retry_after).isdigit():
                sleep_for = max(sleep_for, min(int(retry_after), 300))
            elif exc.code == 429:
                sleep_for = max(sleep_for, 65)
            print(
                f"  HTTP {exc.code}; retry {attempt}/{MAX_RETRIES} in {sleep_for:.1f}s (Retry-After={retry_after})",
                flush=True,
            )
            time.sleep(sleep_for)
            delay *= 2
        except (TimeoutError, URLError, SPARQLWrapperException, OSError, ConnectionError) as exc:
            last_error = exc
            if attempt == MAX_RETRIES:
                raise
            print(f"  {type(exc).__name__}: {exc}; retry {attempt}/{MAX_RETRIES} in {delay:.1f}s", flush=True)
            time.sleep(delay)
            delay *= 2
    raise last_error if last_error else RuntimeError("SPARQL query failed")


def resolve_death_states(death_place_uris: list[str]) -> pd.DataFrame:
    qids = []
    seen: set[str] = set()
    for uri in death_place_uris:
        qid = qid_from_uri(uri)
        if qid and qid not in seen:
            seen.add(qid)
            qids.append(qid)
    if not qids:
        return pd.DataFrame(columns=["deathPlace", "deathStateLabel"])

    frames = []
    for start in range(0, len(qids), PLACE_CHUNK):
        chunk = qids[start : start + PLACE_CHUNK]
        values = " ".join(f"wd:{qid}" for qid in chunk)
        payload = query_wikidata(DEATH_STATE_QUERY % values)
        bindings = payload.get("results", {}).get("bindings", [])
        frames.append(bindings_to_frame(bindings, ("deathPlace", "deathState", "deathStateLabel")))
        if start + PLACE_CHUNK < len(qids):
            time.sleep(RATE_LIMIT_S)
    resolved = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    resolved = resolved.dropna(subset=["deathStateLabel"]).drop_duplicates(subset=["deathPlace"], keep="first")
    return resolved[["deathPlace", "deathStateLabel"]]



QLEVER_SPARQL = "https://qlever.dev/api/wikidata"
QLEVER_QUERY = """
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX xsd: <http://www.w3.org/2001/XMLSchema#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
SELECT ?person ?personLabel ?deathDate ?deathPlace ?deathCountry ?birthLoc ?deathLoc ?deathState WHERE {
  {
    SELECT ?person ?birthPlace ?deathPlace ?deathDate WHERE {
      ?birthPlace wdt:P131* wd:{STATE_QCODE} .
      ?person wdt:P19 ?birthPlace ;
              wdt:P31 wd:Q5 ;
              wdt:P20 ?deathPlace ;
              wdt:P570 ?deathDate .
      FILTER(?deathDate >= "1980-01-01T00:00:00Z"^^xsd:dateTime)
    }
    ORDER BY DESC(?deathDate)
    LIMIT {LIMIT}
  }
  OPTIONAL { ?person rdfs:label ?personLabel . FILTER(LANG(?personLabel) = "en") }
  OPTIONAL { ?deathPlace wdt:P17 ?deathCountry . }
  OPTIONAL { ?birthPlace wdt:P625 ?birthLoc . }
  OPTIONAL { ?deathPlace wdt:P625 ?deathLoc . }
  OPTIONAL {
    ?deathPlace wdt:P131* ?deathState .
    ?deathState wdt:P31 wd:Q35657 .
  }
}
"""


def _point_lat_lon(value: object) -> tuple[float | None, float | None]:
    if not isinstance(value, str) or "(" not in value or ")" not in value:
        return None, None
    inside = value[value.find("(") + 1 : value.find(")")]
    parts = inside.replace(",", " ").split()
    if len(parts) != 2:
        return None, None
    try:
        lon, lat = float(parts[0]), float(parts[1])
    except ValueError:
        return None, None
    return lat, lon


def qlever_frame(state_qid: str, limit: int) -> pd.DataFrame:
    """Same cohort as the WDQS extract, from the public QLever Wikidata index."""
    import urllib.parse
    import urllib.request

    query = QLEVER_QUERY.replace("{STATE_QCODE}", state_qid).replace("{LIMIT}", str(int(limit)))
    url = QLEVER_SPARQL + "?" + urllib.parse.urlencode({"query": query})
    request = urllib.request.Request(
        url,
        headers={
            "Accept": "application/sparql-results+json",
            "User-Agent": USER_AGENT,
        },
    )
    payload = None
    delay = 8.0
    for attempt in range(1, 4):
        try:
            with urllib.request.urlopen(request, timeout=QUERY_TIMEOUT_S) as response:
                payload = json.loads(response.read().decode("utf-8"), strict=False)
            break
        except HTTPError as exc:
            if exc.code not in {429, 500, 502, 503, 504} or attempt == 3:
                raise
            print(f"  QLever HTTP {exc.code}; retry {attempt}/3 in {delay:.0f}s", flush=True)
            time.sleep(delay)
            delay *= 2
    bindings = payload.get("results", {}).get("bindings", []) if payload else []
    rows = []
    qid_to_name = {meta["qid"]: name for name, meta in STATES.items()}
    for binding in bindings:
        value = lambda key: binding.get(key, {}).get("value")
        birth_lat, birth_lon = _point_lat_lon(value("birthLoc"))
        death_lat, death_lon = _point_lat_lon(value("deathLoc"))
        death_state_qid = qid_from_uri(value("deathState"))
        country_qid = qid_from_uri(value("deathCountry"))
        rows.append(
            {
                "person": value("person"),
                "personLabel": value("personLabel"),
                "deathDate": value("deathDate"),
                "deathPlace": value("deathPlace"),
                "deathCountry": value("deathCountry"),
                "deathCountryLabel": "United States" if country_qid == "Q30" else None,
                "birthLat": birth_lat,
                "birthLon": birth_lon,
                "deathLat": death_lat,
                "deathLon": death_lon,
                "deathState": value("deathState"),
                "deathStateLabel": qid_to_name.get(death_state_qid or ""),
            }
        )
    return pd.DataFrame(rows, columns=list(PERSON_COLUMNS))


def cache_path_for(state_name: str, directory: Path = DATA_DIR) -> Path:
    return directory / f"{state_name}.csv"


def fetch_state(
    state_name: str,
    qid: str,
    state_code: str,
    limit: int = FULL_LIMIT,
    directory: Path = DATA_DIR,
    skip_existing: bool = True,
) -> Path | None:
    path = cache_path_for(state_name, directory)
    if skip_existing and path.exists():
        print(f"skip  {state_name}: {path} already exists")
        return path

    print(f"fetch {state_name} ({qid}), LIMIT {limit}", flush=True)
    try:
        payload = query_wikidata(build_query(qid, limit))
        bindings = payload.get("results", {}).get("bindings", [])
        frame = bindings_to_frame(bindings, PERSON_COLUMNS)
    except HTTPError as exc:
        if exc.code != 429:
            raise
        print("  WDQS returned 429; falling back to QLever", flush=True)
        frame = qlever_frame(qid, limit)
    if frame.empty:
        frame["deathStateLabel"] = pd.NA
    elif frame["deathStateLabel"].notna().any():
        # Prefer the copy of a person that actually resolved a US death state.
        frame = frame.sort_values("deathStateLabel", na_position="last", kind="mergesort")
        print(
            f"  death states in extract: {int(frame['deathStateLabel'].notna().sum()):,} of {len(frame):,} rows",
            flush=True,
        )
    else:
        time.sleep(RATE_LIMIT_S)
        places = frame["deathPlace"].dropna().astype(str).tolist()
        print(f"  resolving US death states for {len(set(places))} unique death places", flush=True)
        try:
            resolved = resolve_death_states(places)
            if not resolved.empty:
                frame = frame.merge(resolved, on="deathPlace", how="left")
            else:
                frame["deathStateLabel"] = pd.NA
        except Exception as exc:  # noqa: BLE001 — cache the person extract even if state resolve times out
            print(f"  death-state resolve failed ({type(exc).__name__}: {exc}); continuing without deathStateLabel", flush=True)
            frame["deathStateLabel"] = pd.NA
    frame = frame.drop_duplicates(subset=["person"], keep="first")
    frame["birth_state"] = state_name
    frame["birthStateLabel"] = state_name
    frame["state_code"] = state_code
    if frame.empty:
        raise RuntimeError(f"{state_name} extract returned no rows")
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)
    print(f"  wrote {len(frame):,} rows -> {path}", flush=True)
    return path


def fetch_states(
    states: dict[str, dict[str, str]],
    limit: int = FULL_LIMIT,
    directory: Path = DATA_DIR,
) -> list[str]:
    failures = load_failures()
    failed: list[str] = []
    pending = [(name, meta) for name, meta in states.items() if not cache_path_for(name, directory).exists()]
    for index, (state_name, meta) in enumerate(pending):
        try:
            fetch_state(
                state_name,
                meta["qid"],
                meta["code"],
                limit=limit,
                directory=directory,
                skip_existing=True,
            )
            failures.pop(state_name, None)
        except Exception as exc:  # noqa: BLE001 — resume remaining states after a hard failure
            message = f"{type(exc).__name__}: {exc}"
            print(f"  FAIL {state_name}: {message}")
            failures[state_name] = message
            failed.append(state_name)
        finally:
            save_failures(failures)
            if index < len(pending) - 1:
                time.sleep(RATE_LIMIT_S)
    if failed:
        print(f"Failed states ({len(failed)}): {', '.join(failed)}")
        print(f"Details: {FAILURES_PATH}")
    else:
        print("All requested states cached (or were already on disk).")
    return failed

In [4]:
if RUN_FULL_EXTRACT:
    fetch_states(STATES, limit=FULL_LIMIT, directory=DATA_DIR)
else:
    existing = list(DATA_DIR.glob("*.csv"))
    print(
        f"RUN_FULL_EXTRACT is False. Cached state CSVs in {DATA_DIR}: {len(existing)}. "
        "Set RUN_FULL_EXTRACT = True to query Wikidata for missing states."
    )

All requested states cached (or were already on disk).


### Smoke test

One state, `LIMIT 20`, written under `./data/_smoke/` so it never collides with a full extract. Existing smoke CSVs are skipped; delete that file to refresh. The rest of the notebook falls back to this file if no full-state CSVs exist.

In [5]:
smoke_meta = STATES[SMOKE_STATE]
smoke_path = fetch_state(
    SMOKE_STATE,
    smoke_meta["qid"],
    smoke_meta["code"],
    limit=SMOKE_LIMIT,
    directory=SMOKE_DIR,
    skip_existing=True,
)
smoke_raw = pd.read_csv(smoke_path)
print(f"Smoke test rows: {len(smoke_raw):,}")
display(smoke_raw.head())
assert len(smoke_raw) > 0, "Smoke query returned no rows"
assert smoke_raw["birth_state"].eq(SMOKE_STATE).all()
assert set(["birthLat", "birthLon", "deathLat", "deathLon"]).issubset(smoke_raw.columns)

skip  Rhode Island: data/_smoke/Rhode Island.csv already exists
Smoke test rows: 20


,person,personLabel,deathDate,deathPlace,deathCountry,deathCountryLabel,birthLat,birthLon,deathLat,deathLon,deathState,deathStateLabel,birth_state,birthStateLabel,state_code
0,http://www.wikidata.org/entity/Q106414442,Marge Jacques,2001-01-01T00:00:00Z,http://www.wikidata.org/entity/Q816,http://www.wikidata.org/entity/Q30,United States,42.01,-71.52,34.29,-111.66,http://www.wikidata.org/entity/Q816,Arizona,Rhode Island,Rhode Island,RI
1,http://www.wikidata.org/entity/Q6205537,Joan Walters,2011-07-31T00:00:00Z,http://www.wikidata.org/entity/Q79931,http://www.wikidata.org/entity/Q30,United States,41.82,-71.41,34.54,-112.47,http://www.wikidata.org/entity/Q816,Arizona,Rhode Island,Rhode Island,RI
2,http://www.wikidata.org/entity/Q363867,Frankie Carle,2001-03-07T00:00:00Z,http://www.wikidata.org/entity/Q49261,http://www.wikidata.org/entity/Q30,United States,41.82,-71.41,33.41,-111.83,http://www.wikidata.org/entity/Q816,Arizona,Rhode Island,Rhode Island,RI
3,http://www.wikidata.org/entity/Q454162,Beverly Baker Fleitz,2014-04-29T00:00:00Z,http://www.wikidata.org/entity/Q16739,http://www.wikidata.org/entity/Q30,United States,41.82,-71.41,33.77,-118.20,http://www.wikidata.org/entity/Q99,California,Rhode Island,Rhode Island,RI
4,http://www.wikidata.org/entity/Q16017342,Dean Brelis,2006-11-17T00:00:00Z,http://www.wikidata.org/entity/Q47164,http://www.wikidata.org/entity/Q30,United States,41.49,-71.31,34.02,-118.48,http://www.wikidata.org/entity/Q99,California,Rhode Island,Rhode Island,RI


## Section 2: Consolidation & Vectorized Feature Engineering

Load cached CSVs, drop duplicate person URIs, coerce coordinates, then compute great-circle distance with a vectorized NumPy haversine.

`left_birth_state` is 1 if the person died outside the United States, or died in a resolved US state other than their birth state. Domestic rows with no resolvable death state are **not** counted as leavers; they are flagged in `unresolved_death_state` and excluded from the out-migration denominator.

In [6]:
def haversine_vectorized(lat1, lon1, lat2, lon2):
    """Calculates Great-Circle distance in kilometers between NumPy arrays of coordinates."""
    r = 6371.0  # Earth's radius in kilometers
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = np.sin(dlat / 2.0) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    return 2 * r * np.arcsin(np.sqrt(a))


USA_LABELS = {"United States", "United States of America"}
USA_QID_SUFFIX = "/Q30"
COORD_COLS = ["birthLat", "birthLon", "deathLat", "deathLon"]


def list_state_csvs() -> list[Path]:
    full = sorted(p for p in DATA_DIR.glob("*.csv") if not p.name.startswith("_"))
    if full:
        return full
    smoke = sorted(SMOKE_DIR.glob("*.csv"))
    if smoke:
        print("No full-state cache found; using smoke-test CSV(s) for sections 2–4.")
        return smoke
    raise FileNotFoundError(
        f"No CSVs in {DATA_DIR} or {SMOKE_DIR}. Run the smoke-test cell or set RUN_FULL_EXTRACT = True."
    )


def add_mobility_features(frame: pd.DataFrame) -> pd.DataFrame:
    df = frame.copy()
    for col in COORD_COLS:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Keep rows without coordinates. Stay/leave rates and death totals do not need them;
    # distance is filled only where both places are geocoded.
    coord_ok = df[COORD_COLS].notna().all(axis=1)
    distance_km = np.full(len(df), np.nan)
    if bool(coord_ok.any()):
        distance_km[coord_ok.to_numpy()] = haversine_vectorized(
            df.loc[coord_ok, "birthLat"].to_numpy(),
            df.loc[coord_ok, "birthLon"].to_numpy(),
            df.loc[coord_ok, "deathLat"].to_numpy(),
            df.loc[coord_ok, "deathLon"].to_numpy(),
        )
    df["distance_km"] = distance_km
    df["distance_miles"] = df["distance_km"] * 0.621371

    death_country_label = df["deathCountryLabel"]
    death_country_uri = df["deathCountry"] if "deathCountry" in df.columns else pd.Series(pd.NA, index=df.index)
    death_state = df["deathStateLabel"]
    birth_state = df["birth_state"]

    has_country = death_country_uri.notna() | death_country_label.notna()
    is_usa = death_country_uri.fillna("").astype(str).str.endswith(USA_QID_SUFFIX) | death_country_label.isin(
        USA_LABELS
    )
    is_international = has_country & ~is_usa
    has_us_death_state = death_state.notna() & (death_state.astype(str).str.strip() != "")
    unresolved = ~is_international & ~has_us_death_state
    left = is_international | (has_us_death_state & (death_state != birth_state))

    df["emigrated_internationally"] = is_international.astype(int)
    df["unresolved_death_state"] = unresolved.astype(int)
    df["left_birth_state"] = left.astype(int)
    df["in_outmigration_denominator"] = (~unresolved).astype(int)
    return df


csv_files = list_state_csvs()
master = pd.concat((pd.read_csv(path) for path in csv_files), ignore_index=True)
before = len(master)
master = master.drop_duplicates(subset=["person"], keep="first")
print(f"Loaded {before:,} rows from {len(csv_files)} file(s); {len(master):,} after de-duplicating person URIs.")

df = add_mobility_features(master)
df["deathDate"] = pd.to_datetime(df["deathDate"], utc=True, errors="coerce")
print(
    f"Rows: {len(df):,} | "
    f"geocoded: {int(df['distance_km'].notna().sum()):,} | "
    f"denominator: {int(df['in_outmigration_denominator'].sum()):,} | "
    f"unresolved death state: {int(df['unresolved_death_state'].sum()):,}"
)
display(df[["personLabel", "birth_state", "deathStateLabel", "deathCountryLabel", "distance_km", "left_birth_state"]].head())

Loaded 67,578 rows from 50 file(s); 67,465 after de-duplicating person URIs.


Rows: 67,465 | geocoded: 67,331 | denominator: 66,264 | unresolved death state: 1,201


,personLabel,birth_state,deathStateLabel,deathCountryLabel,distance_km,left_birth_state
0,Jimmy Lee Sudduth,Alabama,Alabama,United States,0.00,0
1,Ron Billingsley,Alabama,Alabama,United States,176.32,0
2,Jimmy Holley,Alabama,Alabama,United States,8.75,0
3,Jim Clark,Alabama,Alabama,United States,0.00,0
4,Sherrill Nielsen,Alabama,Alabama,United States,207.53,0


## Section 3: Statistical Analysis & Binarization

State-level summary:

- `total_records` — rows in the out-migration denominator
- `mean_distance_km` / `median_distance_km` — all geocoded rows
- `outmigration_rate` — percent of the denominator who left their birth state
- `is_high_outmigration_state` — rate above the **unweighted** mean of state rates (each state counts equally under the 5,000 cap)
- `is_above_national_median_dist` — state median above the pooled person-level national median of `distance_km`

Destination table: top US death states among people who left for another resolved US state. International deaths stay in the emigration rate and are not mixed into that ranking.

In [7]:
def build_state_summary(frame: pd.DataFrame) -> pd.DataFrame:
    denom = frame["in_outmigration_denominator"].eq(1)
    summary = (
        frame.groupby(["birth_state", "state_code"], dropna=False)
        .agg(
            geocoded_records=("distance_km", "count"),
            mean_distance_km=("distance_km", "mean"),
            median_distance_km=("distance_km", "median"),
            unresolved_death_state=("unresolved_death_state", "sum"),
        )
        .reset_index()
    )
    denom_stats = (
        frame.loc[denom]
        .groupby(["birth_state", "state_code"], dropna=False)
        .agg(
            total_records=("person", "size"),
            leavers=("left_birth_state", "sum"),
            international_deaths=("emigrated_internationally", "sum"),
        )
        .reset_index()
    )
    summary = summary.merge(denom_stats, on=["birth_state", "state_code"], how="left")
    summary["total_records"] = summary["total_records"].fillna(0).astype(int)
    summary["leavers"] = summary["leavers"].fillna(0).astype(int)
    summary["international_deaths"] = summary["international_deaths"].fillna(0).astype(int)
    summary["unresolved_death_state"] = summary["unresolved_death_state"].astype(int)
    summary["outmigration_rate"] = np.where(
        summary["total_records"] > 0,
        100.0 * summary["leavers"] / summary["total_records"],
        np.nan,
    )

    national_avg_rate = summary["outmigration_rate"].mean()
    national_median_dist = frame["distance_km"].median()
    summary["is_high_outmigration_state"] = summary["outmigration_rate"] > national_avg_rate
    summary["is_above_national_median_dist"] = summary["median_distance_km"] > national_median_dist
    summary = summary.sort_values("outmigration_rate", ascending=False).reset_index(drop=True)
    return summary, float(national_avg_rate), float(national_median_dist)


def us_outmigrant_destinations(frame: pd.DataFrame) -> pd.DataFrame:
    moved = frame[
        frame["deathStateLabel"].notna()
        & (frame["deathStateLabel"] != frame["birth_state"])
        & frame["emigrated_internationally"].eq(0)
    ]
    if moved.empty:
        return pd.DataFrame(columns=["birth_state", "death_state", "n", "rank"])
    counts = (
        moved.groupby(["birth_state", "deathStateLabel"], dropna=False)
        .size()
        .reset_index(name="n")
        .sort_values(["birth_state", "n"], ascending=[True, False])
    )
    counts["rank"] = counts.groupby("birth_state")["n"].rank(method="first", ascending=False).astype(int)
    return counts.rename(columns={"deathStateLabel": "death_state"})


state_summary, national_avg_rate, national_median_dist = build_state_summary(df)
destinations = us_outmigrant_destinations(df)
primary_destination = (
    destinations.loc[destinations["rank"] == 1, ["birth_state", "death_state", "n"]]
    .rename(columns={"death_state": "top_destination", "n": "top_destination_count"})
)
state_summary = state_summary.merge(primary_destination, on="birth_state", how="left")

print(
    "Unweighted mean of state out-migration rates: "
    f"{national_avg_rate:.2f}%"
)
print(
    "Pooled national median birth-to-death distance: "
    f"{national_median_dist:.1f} km"
)
display(
    state_summary[
        [
            "birth_state",
            "state_code",
            "total_records",
            "mean_distance_km",
            "median_distance_km",
            "outmigration_rate",
            "is_high_outmigration_state",
            "is_above_national_median_dist",
            "unresolved_death_state",
            "top_destination",
        ]
    ]
)
print("Top 3 US death destinations per birth state")
display(destinations.loc[destinations["rank"] <= 3])

Unweighted mean of state out-migration rates: 69.26%
Pooled national median birth-to-death distance: 756.3 km


,birth_state,state_code,total_records,mean_distance_km,median_distance_km,outmigration_rate,is_high_outmigration_state,is_above_national_median_dist,unresolved_death_state,top_destination
0,Nebraska,NE,785,"1,501.42","1,663.62",84.46,True,True,13,California
1,Idaho,ID,309,"1,379.32",934.79,83.50,True,True,4,California
2,Iowa,IA,1356,"1,419.07","1,364.18",82.01,True,True,24,California
3,Kansas,KS,933,"1,497.66","1,579.82",81.89,True,True,13,California
4,Colorado,CO,682,"1,600.22","1,335.95",81.23,True,True,9,California
5,North Dakota,ND,317,"1,464.53","1,643.99",81.07,True,True,2,California
6,Wyoming,WY,158,"1,317.62","1,160.48",79.75,True,True,2,California
7,Missouri,MO,1915,"1,465.38","1,368.99",79.63,True,True,35,California
8,West Virginia,WV,558,"1,089.94",508.46,78.14,True,False,15,California
9,Illinois,IL,4792,"1,602.56","1,177.22",77.69,True,True,84,California


Top 3 US death destinations per birth state


,birth_state,death_state,n,rank
3,Alabama,California,101,1
7,Alabama,Florida,78,2
28,Alabama,New York,70,3
60,Alaska,Washington,16,1
45,Alaska,California,15,2
...,...,...,...,...
1955,Wisconsin,Florida,70,2
1978,Wisconsin,New York,67,3
1995,Wyoming,California,27,1
1996,Wyoming,Colorado,18,2


## Section 4: United States maps

Two choropleths of this Wikidata cohort (notable people, not a resident sample):

1. **Tendency to die in the same state you were born.** For each birth state, the percent of resolved deaths that happened in that same state. International deaths count as not same-state. Rows with no resolvable US death state and no foreign country are left out of the rate.
2. **US native deaths per state.** How many people in this extract died in each state, including people who died where they were born. A state with no resolved deaths in the cache is drawn as zero.


In [8]:
import plotly.io as pio

pio.renderers.default = "plotly_mimetype"
import plotly.express as px

stay = state_summary.copy()
stay["same_state_death_rate"] = np.where(
    stay["total_records"] > 0,
    100.0 * (stay["total_records"] - stay["leavers"]) / stay["total_records"],
    np.nan,
)

fig_stay = px.choropleth(
    stay.dropna(subset=["same_state_death_rate"]),
    locations="state_code",
    locationmode="USA-states",
    color="same_state_death_rate",
    scope="usa",
    color_continuous_scale="Blues",
    range_color=(0, 100),
    title="Tendency to die in the same state you were born",
    labels={"same_state_death_rate": "Same-state deaths (%)"},
    hover_name="birth_state",
    hover_data={
        "state_code": False,
        "total_records": True,
        "same_state_death_rate": ":.1f",
    },
)
fig_stay.update_layout(margin=dict(l=0, r=0, t=60, b=0))
maps_dir = DATA_DIR / "maps"
maps_dir.mkdir(parents=True, exist_ok=True)
fig_stay.write_html(maps_dir / "same_state_death_rate.html", include_plotlyjs=True)
fig_stay.show()


In [9]:
state_names = {name.casefold(): name for name in STATES}
state_codes = {name.casefold(): meta["code"] for name, meta in STATES.items()}
labels = (
    df["deathStateLabel"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.casefold()
    .str.removeprefix("state of ")
    .str.strip()
)
domestic = pd.DataFrame(
    {
        "death_state": labels.map(state_names),
        "state_code": labels.map(state_codes),
    }
).dropna()
counted = (
    domestic.groupby(["death_state", "state_code"], as_index=False)
    .size()
    .rename(columns={"size": "native_deaths"})
)
all_states = pd.DataFrame(
    [{"death_state": name, "state_code": meta["code"]} for name, meta in STATES.items()]
)
death_totals = all_states.merge(counted, on=["death_state", "state_code"], how="left")
death_totals["native_deaths"] = death_totals["native_deaths"].fillna(0).astype(int)

fig_deaths = px.choropleth(
    death_totals,
    locations="state_code",
    locationmode="USA-states",
    color="native_deaths",
    scope="usa",
    color_continuous_scale="YlOrRd",
    title="US native deaths per state",
    labels={"native_deaths": "Deaths"},
    hover_name="death_state",
    hover_data={"state_code": False, "native_deaths": True},
)
fig_deaths.update_layout(margin=dict(l=0, r=0, t=60, b=0))
fig_deaths.write_html(maps_dir / "us_native_deaths_by_state.html", include_plotlyjs=True)
fig_deaths.show()


In [10]:
print("Same-state death rate by birth state")
display(
    stay.sort_values("same_state_death_rate", ascending=False)[
        ["birth_state", "state_code", "total_records", "leavers", "same_state_death_rate"]
    ]
)
print("US native death totals by state")
display(death_totals.sort_values(["native_deaths", "death_state"], ascending=[False, True]))


Same-state death rate by birth state


,birth_state,state_code,total_records,leavers,same_state_death_rate
49,California,CA,4843,1932,60.11
48,Alaska,AK,144,59,59.03
47,Nevada,NV,189,81,57.14
46,Hawaii,HI,338,167,50.59
45,Louisiana,LA,1448,771,46.75
44,Utah,UT,640,342,46.56
43,Texas,TX,3319,1825,45.01
42,North Carolina,NC,1284,726,43.46
41,Florida,FL,1004,569,43.33
40,New Mexico,NM,252,158,37.30


US native death totals by state


,death_state,state_code,native_deaths
4,California,CA,13188
31,New York,NY,5374
8,Florida,FL,3774
42,Texas,TX,3180
37,Pennsylvania,PA,2322
12,Illinois,IL,2252
20,Massachusetts,MA,2149
45,Virginia,VA,1802
19,Maryland,MD,1616
29,New Jersey,NJ,1598
